# 01 — Analytics Ingestion

Validate growth-agent's analytics ingestion against real Scaleway S3 data, entirely
inside the `growth-agent-dev/` sandbox prefix:
- Per-post engagement metrics (`_collect_post_metrics`)
- Page-traffic from the self-hosted `analytics` service's monthly rollups (`_collect_page_traffic`)

Umami Cloud is gone — it was fully replaced this session by the self-hosted `analytics`
service (a separate package in this monorepo that beacons pageviews straight to S3, no
external API). `agent/umami_client.py` no longer exists and `WebsiteAnalytics` no longer
exists in `agent/models.py`, so this notebook no longer touches either.

In [1]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env', override=True)

import os
prefix = os.getenv("S3_STATE_PREFIX", "growth-agent/")
if prefix == "growth-agent/":
    raise RuntimeError(
        f"S3_STATE_PREFIX is {prefix!r} — this is the PRODUCTION prefix. "
        "Set S3_STATE_PREFIX=growth-agent-dev/ in .env before running notebooks."
    )
print(f"Using S3 prefix: {prefix}  ✓")


Using S3 prefix: growth-agent-dev/  ✓


In [2]:
from agent.storage import S3Storage

store = S3Storage(
    bucket=os.getenv('S3_BUCKET'),
    prefix=os.getenv('S3_STATE_PREFIX'),
    access_key=os.getenv('SCW_ACCESS_KEY'),
    secret_key=os.getenv('SCW_SECRET_KEY'),
)
print(f"store targets s3://{store.bucket}/{store.prefix}")


store targets s3://my-imagestore/growth-agent-dev/


## 1. Pull real published-post data into the dev sandbox

Copy the production `content_queue.json` into the dev prefix (read-only from prod).
Done once here — shared by the per-post-metrics section below and the page-traffic
section further down, so real published-post links (and their real analytics traffic)
are visible in both.

In [3]:
# Copy prod content_queue.json → dev prefix (prod is read-only here)
import boto3

prod_prefix = os.environ["S3_STATE_PREFIX_PROD"]  # e.g. "growth-agent/"
dev_prefix = os.getenv("S3_STATE_PREFIX", "growth-agent-dev/")
bucket = os.environ["S3_BUCKET"]

s3_raw = boto3.client(
    "s3",
    region_name="nl-ams",
    endpoint_url="https://s3.nl-ams.scw.cloud",
    aws_access_key_id=os.environ["SCW_ACCESS_KEY"],
    aws_secret_access_key=os.environ["SCW_SECRET_KEY"],
)
s3_raw.copy_object(
    Bucket=bucket,
    CopySource={"Bucket": bucket, "Key": prod_prefix + "content_queue.json"},
    Key=dev_prefix + "content_queue.json",
)
print(f"Copied {prod_prefix}content_queue.json → {dev_prefix}content_queue.json")


Copied growth-agent/content_queue.json → growth-agent-dev/content_queue.json


## 2. Per-post engagement metrics

In [4]:
from agent.nodes.ingest import _collect_post_metrics

_collect_post_metrics(store)
print("Done — performance.json written to dev prefix")


Done — performance.json written to dev prefix


In [5]:
from agent.storage import load_model
from agent.models import Performance

perf = load_model(store, "performance.json", Performance)
print(f"Posts with metrics: {len(perf.posts)}")
for p in perf.posts:
    print(f"  [{p.channel:8}] ❤️{p.favourites:3}  🔁{p.reblogs:3}  💬{p.replies:3}  — {p.id[:24]}…")


Posts with metrics: 59
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  1  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  1  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  1  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  

## 3. Inspect the real analytics rollups directly (read-only)

The `analytics` service (a separate package) writes monthly rollups to
`rollup/fretchen.eu/{YYYY-MM}.json` at the **bucket root** — not under any
`growth-agent[-dev]/` prefix. This read is always real production analytics data and is
safe regardless of `S3_STATE_PREFIX`, because it never writes anything.

This mirrors exactly what `_collect_page_traffic` does internally, reusing its own
`_sum_trailing_hits` helper rather than reimplementing the month-boundary summation.

In [6]:
from datetime import datetime, timedelta, timezone
from agent.nodes.ingest import _sum_trailing_hits, PAGE_TRAFFIC_WINDOW_DAYS, ANALYTICS_SITE

analytics_storage = S3Storage(
    bucket=os.environ["S3_BUCKET"],
    prefix="",
    access_key=os.environ["SCW_ACCESS_KEY"],
    secret_key=os.environ["SCW_SECRET_KEY"],
)

today = datetime.now(timezone.utc).date()
window_start = today - timedelta(days=PAGE_TRAFFIC_WINDOW_DAYS - 1)
months = sorted({window_start.strftime("%Y-%m"), today.strftime("%Y-%m")})
rollups = {}
for month in months:
    data = analytics_storage.read(f"rollup/{ANALYTICS_SITE}/{month}.json")
    rollups[month] = data if isinstance(data, dict) else {"days": {}}

hits_by_page = _sum_trailing_hits(rollups, window_start, today)

print(f"Window: {window_start} .. {today}  ({len(hits_by_page)} pages)")
for path, hits in sorted(hits_by_page.items(), key=lambda kv: -kv[1])[:15]:
    print(f"  {hits:>6}  {path}")

print()
print(f"/x402/ (read-only, real rollups): {hits_by_page.get('/x402/')}")


Window: 2026-07-20 .. 2026-08-18  (39 pages)
      61  /
      17  /blog/16/
       4  /quantum/hardware/2/
       4  /quantum/amo/
       3  /blog/29/
       3  /blog/25/
       3  /blog/7/
       2  /blog/9/
       2  /quantum/amo/18/
       2  /quantum/amo/12/
       2  /notebook-smoke-test
       2  /analytics/
       2  /blog/23/
       2  /blog/24/
       2  /blog/27/

/x402/ (read-only, real rollups): None


## 4. Run `_collect_page_traffic` against the dev prefix

Merges into the dev prefix's `performance.json`, preserving the `posts` written in
section 2 above — the same safe write order `ingest_analytics()` uses in production
(page traffic is always collected *after* post metrics, since `_collect_post_metrics`
does a fresh, non-merging write).

In [7]:
from agent.nodes.ingest import _collect_page_traffic

_collect_page_traffic(store)
print("Done — page_traffic merged into performance.json (dev prefix)")


Done — page_traffic merged into performance.json (dev prefix)


## 5. Verify: reload `performance.json` from the dev prefix

In [8]:
perf = load_model(store, "performance.json", Performance)

print(f"Posts still present: {len(perf.posts)}")
print(f"page_traffic: {len(perf.page_traffic)} pages")
for path, hits in sorted(perf.page_traffic.items(), key=lambda kv: -kv[1])[:15]:
    print(f"  {hits:>6}  {path}")

print()
print(f"/x402/ (from dev performance.json): {perf.page_traffic.get('/x402/')}")
print("Should match the read-only figure printed in section 3.")


Posts still present: 59
page_traffic: 39 pages
      61  /
      17  /blog/16/
       4  /quantum/hardware/2/
       4  /quantum/amo/
       3  /blog/29/
       3  /blog/25/
       3  /blog/7/
       2  /blog/9/
       2  /quantum/amo/18/
       2  /quantum/amo/12/
       2  /notebook-smoke-test
       2  /analytics/
       2  /blog/23/
       2  /blog/24/
       2  /blog/27/

/x402/ (from dev performance.json): None
Should match the read-only figure printed in section 3.
